<a href="https://colab.research.google.com/github/banerjee109/Infotact-Project-2/blob/main/cohort_retention_cltv_analysis_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(7)
random.seed(7)

# ============================================================
# CloudSync SaaS — Subscription Billing Dataset
# Simulates 18 months of subscription history for a B2B SaaS product
# ============================================================

NUM_CUSTOMERS = 2500
START_DATE = datetime(2023, 1, 1)
NUM_MONTHS = 18

PLAN_TIERS = {
    'Starter':    {'mrr': 15,  'churn_risk': 0.14},   # cheap plan -> churns faster
    'Growth':     {'mrr': 49,  'churn_risk': 0.08},
    'Scale':      {'mrr': 129, 'churn_risk': 0.05},
    'Enterprise': {'mrr': 399, 'churn_risk': 0.02},   # sticky, high-touch customers
}

CHANNELS = ['Organic_Search', 'Paid_Ads', 'Referral', 'Outbound_Sales', 'Partner_Integration']
INDUSTRIES = ['E-commerce', 'FinTech', 'Healthcare', 'EdTech', 'Manufacturing', 'Agency']
CANCEL_REASONS = ['Too_Expensive', 'Missing_Features', 'Switched_Competitor', 'Poor_Support', 'No_Longer_Needed', None]

print("Simulating CloudSync subscription lifecycle...")

rows = []
customer_num = 1000

for _ in range(NUM_CUSTOMERS):
    customer_id = f'CS_{customer_num}'
    customer_num += 1

    # Weighted signup month -> more customers joined earlier (natural for cohort spread)
    signup_offset = np.random.choice(
        range(NUM_MONTHS - 2),
        p=np.array([1/(m+1)**0.5 for m in range(NUM_MONTHS - 2)]) /
          sum(1/(m+1)**0.5 for m in range(NUM_MONTHS - 2))
    )
    signup_date = START_DATE + timedelta(days=int(signup_offset * 30 + random.randint(0, 27)))

    plan = random.choices(list(PLAN_TIERS.keys()), weights=[45, 30, 18, 7])[0]
    channel = random.choices(CHANNELS, weights=[30, 25, 20, 15, 10])[0]
    industry = random.choice(INDUSTRIES)
    company_size = random.choice(['1-10', '11-50', '51-200', '201-500', '500+'])

    base_churn = PLAN_TIERS[plan]['churn_risk']
    mrr = PLAN_TIERS[plan]['mrr']

    is_active = True
    current_month = 0
    cancel_reason = None
    plan_history = plan

    while is_active and (signup_offset + current_month) < NUM_MONTHS:
        billing_date = signup_date + timedelta(days=30 * current_month)

        # Occasional upgrade after month 3 (happy customers expanding)
        if current_month == 3 and plan != 'Enterprise' and random.random() < 0.12:
            tiers = list(PLAN_TIERS.keys())
            idx = tiers.index(plan_history)
            plan_history = tiers[min(idx + 1, len(tiers) - 1)]
            mrr = PLAN_TIERS[plan_history]['mrr']

        rows.append({
            'customer_id': customer_id,
            'billing_date': billing_date.strftime('%Y-%m-%d'),
            'signup_date': signup_date.strftime('%Y-%m-%d'),
            'plan_tier': plan_history,
            'mrr_usd': mrr,
            'acquisition_channel': channel,
            'industry': industry,
            'company_size': company_size,
            'event_type': 'renewal' if current_month > 0 else 'new_signup',
            'cancellation_reason': None
        })

        # Determine churn — risk grows slightly over time (fatigue), but drops after month 6 (loyal core)
        month_factor = 1.3 if current_month < 3 else (1.0 if current_month < 6 else 0.6)
        churn_prob = base_churn * month_factor

        if random.random() < churn_prob:
            is_active = False
            cancel_reason = random.choices(
                CANCEL_REASONS,
                weights=[30, 25, 20, 15, 10, 0]
            )[0]
            rows.append({
                'customer_id': customer_id,
                'billing_date': (billing_date + timedelta(days=30)).strftime('%Y-%m-%d'),
                'signup_date': signup_date.strftime('%Y-%m-%d'),
                'plan_tier': plan_history,
                'mrr_usd': 0,
                'acquisition_channel': channel,
                'industry': industry,
                'company_size': company_size,
                'event_type': 'cancellation',
                'cancellation_reason': cancel_reason
            })

        current_month += 1

df = pd.DataFrame(rows)
df = df.sample(frac=1, random_state=7).reset_index(drop=True)  # shuffle like a real export

df.to_csv('cloudsync_subscriptions.csv', index=False)
print(f"✅ cloudsync_subscriptions.csv created — {len(df)} rows, {NUM_CUSTOMERS} customers")
print(df.head(10))
df = pd.read_csv('cloudsync_subscriptions.csv')
df.info()
df['event_type'].value_counts()
from google.colab import files
##files.download('cloudsync_subscriptions.csv')

Simulating CloudSync subscription lifecycle...
✅ cloudsync_subscriptions.csv created — 19422 rows, 2500 customers
  customer_id billing_date signup_date   plan_tier  mrr_usd  \
0     CS_2363   2023-03-24  2023-02-22       Scale      129   
1     CS_1564   2023-03-02  2023-01-01     Starter       15   
2     CS_2215   2024-02-28  2023-10-01     Starter       15   
3     CS_2261   2024-03-29  2023-01-04     Starter       15   
4     CS_2525   2023-12-17  2023-12-17     Starter       15   
5     CS_3382   2023-05-13  2023-05-13     Starter       15   
6     CS_2113   2023-02-25  2023-01-26  Enterprise      399   
7     CS_2879   2024-03-04  2024-02-03     Starter        0   
8     CS_2768   2024-05-02  2023-09-05      Growth       49   
9     CS_3216   2024-02-14  2023-08-18       Scale      129   

   acquisition_channel    industry company_size    event_type  \
0             Paid_Ads     FinTech        11-50       renewal   
1             Referral     FinTech         500+       renewal 

In [2]:
import pandas as pd
import numpy as np
import datetime as dt
pd.set_option('display.max_rows' ,None)
pd.set_option('display.max_columns' ,None)
df=pd.read_csv("cloudsync_subscriptions.csv")
df.info()
df.describe()
df.head()
df.columns
df.columns.nunique()
df.isna().sum()
df["customer_id"].nunique()
len(df["customer_id"])
df["company_size"].idxmax()
df["billing_date"].head()
df["billing_date"]=pd.to_datetime(df["billing_date"])
df["billing_date"].head()
df["billing_day"]=df["billing_date"].dt.day
df["billing_month"]=df["billing_date"].dt.month
df["billing_year"]=df["billing_date"].dt.year
df["billing_day_name"]=df["billing_date"].dt.day_name()
df["billing_day_name"].unique()
df["billing_year"].unique()
df["billing_month"].nunique()
missing_percent = (df.isna().sum() / len(df))*100
missing_percent
df.head(1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19422 entries, 0 to 19421
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          19422 non-null  object
 1   billing_date         19422 non-null  object
 2   signup_date          19422 non-null  object
 3   plan_tier            19422 non-null  object
 4   mrr_usd              19422 non-null  int64 
 5   acquisition_channel  19422 non-null  object
 6   industry             19422 non-null  object
 7   company_size         19422 non-null  object
 8   event_type           19422 non-null  object
 9   cancellation_reason  1546 non-null   object
dtypes: int64(1), object(9)
memory usage: 1.5+ MB


,customer_id,billing_date,signup_date,plan_tier,mrr_usd,acquisition_channel,industry,company_size,event_type,cancellation_reason,billing_day,billing_month,billing_year,billing_day_name
0,CS_2363,2023-03-24,2023-02-22,Scale,129,Paid_Ads,FinTech,11-50,renewal,NaN,24,3,2023,Friday


In [43]:
df.head(5)
df.groupby("acquisition_channel")["mrr_usd"].sum()
df.groupby("billing_month")["mrr_usd"].sum()
df.tail(2)
df["cancellation_reason"].dropna()
df.head()
df.sort_values(["customer_id","billing_date"])
first_acquisition= df.groupby("customer_id")["billing_date"].first().reset_index()
last_acquisition= df.groupby("customer_id")["billing_date"].last().reset_index()
df.head()
df.groupby("industry")["mrr_usd"].sum()
df.groupby("billing_day_name")["mrr_usd"].mean()
df.groupby(["industry","billing_day_name"])["mrr_usd"].sum()
##New customers every month
df[(df["event_type"]=="new_signup")]["customer_id"].nunique()
##df["signup_month"]=pd.to_datetime(df["signup_date"]).dt.to_period("M")
df["signup_month"] = pd.to_datetime(df["signup_date"]).dt.to_period("M")
df["signup_month"].nunique()
monthly_new_signups= df[(df["event_type"]=="new_signup")].groupby("signup_month")["customer_id"].nunique()
monthly_new_signups
df["cancellation_reason"].unique()
df.groupby("plan_tier")["mrr_usd"].sum().sort_values(ascending=False).reset_index()
df.groupby("industry")["mrr_usd"].sum().sort_values(ascending=False).reset_index()
df.groupby("company_size")["mrr_usd"].sum().sort_values(ascending=False).reset_index()
df.groupby("event_type")["customer_id"].nunique()


,customer_id
event_type,
cancellation,1546
new_signup,2500
renewal,2194


In [11]:
df.head()
df.columns
df.shape


(19422, 15)